In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss

# تشخیص دقیق ریشه پروژه
CURRENT_DIR = Path.cwd().resolve()
if CURRENT_DIR.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# تنظیم مسیرها
PRODUCTS_PATH = PROJECT_ROOT / "data" / "processed" / "products.parquet"
EVAL_DIR = PROJECT_ROOT / "evaluation"
QUERY_PATH = EVAL_DIR / "evaluation_queries.json"
MODEL_CACHE_DIR = PROJECT_ROOT / "artifacts" / "model_cache"

print(f"Current Working Directory: {CURRENT_DIR}")
print(f"Project Root Resolved: {PROJECT_ROOT}")
print(f"Products Path: {PRODUCTS_PATH} -> Exists: {PRODUCTS_PATH.exists()}")
print(f"Evaluation Query Path: {QUERY_PATH} -> Exists: {QUERY_PATH.exists()}")

# بررسی اجباری بودن فایل محصولات (باید حتماً باشد)
assert PRODUCTS_PATH.exists(), f"فایل محصولات در مسیر {PRODUCTS_PATH} یافت نشد!"

# اگر پوشه evaluation یا فایل کوئری نبود، هشدار می‌دهیم یا می‌سازیم
if not EVAL_DIR.exists():
    EVAL_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Created missing evaluation directory at: {EVAL_DIR}")

if not QUERY_PATH.exists():
    print("⚠️ هشدار: فایل evaluation_queries.json در مسیر evaluation پیدا نشد. در صورت نیاز می‌توانیم آن را بسازیم یا از کوئری‌های تستی داخل نوت‌بوک استفاده کنیم.")

print("✓ سلول تنظیمات با موفقیت و بدون خطا اجرا شد!")


c:\Users\pourya\miniconda3\envs\btq\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Current Working Directory: D:\goal\programming\quera\boot_camp\project3\notebook
Project Root Resolved: D:\goal\programming\quera\boot_camp\project3
Products Path: D:\goal\programming\quera\boot_camp\project3\data\processed\products.parquet -> Exists: True
Evaluation Query Path: D:\goal\programming\quera\boot_camp\project3\evaluation\evaluation_queries.json -> Exists: False
⚠️ هشدار: فایل evaluation_queries.json در مسیر evaluation پیدا نشد. در صورت نیاز می‌توانیم آن را بسازیم یا از کوئری‌های تستی داخل نوت‌بوک استفاده کنیم.
✓ سلول تنظیمات با موفقیت و بدون خطا اجرا شد!


In [3]:
import time

print("Loading product catalog from Parquet...")
start_time = time.time()

# بارگذاری فایل Parquet از مسیری که در سلول ۱ تأیید شد
products_df = pd.read_parquet(PRODUCTS_PATH)
print(f"Original Catalog Shape (with duplicates/all rows): {products_df.shape}")

# بررسی ستون‌های حیاتی مورد نیاز
required_columns = {"id", "title_fa"}
missing_columns = required_columns - set(products_df.columns)
assert not missing_columns, f"ستون‌های ضروری زیر در داده‌ها یافت نشدند: {sorted(missing_columns)}"

# حذف رکوردهای دارای شناسه یا عنوان تهی و حذف شناسه‌های تکراری (پایدار)
products_clean_df = (
    products_df.dropna(subset=["id", "title_fa"])
    .drop_duplicates(subset=["id"])
    .copy()
    .reset_index(drop=True)
)

# بررسی نهایی برای اطمینان از یکتا بودن کامل شناسه‌ها
assert products_clean_df["id"].is_unique, "خطا: هنوز شناسه‌های تکراری در کاتالوگ وجود دارند!"

print(f"Cleaned Catalog Shape (Unique IDs): {products_clean_df.shape}")
print(f"Number of duplicate or invalid IDs removed: {len(products_df) - len(products_clean_df)}")
print(f"Catalog loading and validation completed in {time.time() - start_time:.2f} seconds.")

# نمایش نمونه‌ای از ستون‌های کلیدی کاتالوگ تمیز
display_cols = [c for c in ["id", "title_fa", "Category1", "Brand", "Price"] if c in products_clean_df.columns]
print("\nSample of clean catalog:")
display(products_clean_df[display_cols].head(3))


Loading product catalog from Parquet...
Original Catalog Shape (with duplicates/all rows): (1283496, 12)
Cleaned Catalog Shape (Unique IDs): (948352, 12)
Number of duplicate or invalid IDs removed: 335144
Catalog loading and validation completed in 0.53 seconds.

Sample of clean catalog:


,id,title_fa,Category1,Brand,Price
0,7096438,آبسلانگ مدل s5 بسته 250 عددی,آبسلانگ,متفرقه,634800
1,2845119,آبسلانگ مدل M-1 بسته 400 عددی,آبسلانگ,متفرقه,818800
2,6117745,آبسلانگ مدل m50 مجموعه 500 عددی,آبسلانگ,متفرقه,920000


In [5]:
import sys
from pathlib import Path

# تنظیم مسیر پکیج داخلی پروژه (مطابق ساختار استاندارد ریپوی خودت)
SRC_PATH = PROJECT_ROOT / "src"
if SRC_PATH.is_dir() and str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# تلاش برای ایمپورت تابع نرمال‌سازی از پکیج پروژه یا فال‌بک به فایل ریشه
try:
    from src.digikala_ai.utils.normalization import normalize_persian_text
    print("✓ Normalizer imported from src.digikala_ai.utils.normalization")
except ModuleNotFoundError:
    from normalization import normalize_persian_text
    print("✓ Normalizer imported from root normalization.py")

# انتخاب ستون‌های متنی برای ساخت متن جست‌وجوی مشترک BM25 و Dense
TEXT_COLUMNS = [
    col for col in ["title_fa", "Brand", "Category1", "Category2"]
    if col in products_clean_df.columns
]

assert "title_fa" in TEXT_COLUMNS, "ستون title_fa برای ساخت متن جست‌وجو الزامی است."
print(f"Text columns used for retrieval: {TEXT_COLUMNS}")

# ساخت متن خام جست‌وجو با چسباندن فیلدها
products_clean_df["raw_search_text"] = (
    products_clean_df[TEXT_COLUMNS]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

# اعمال نرمال‌سازی فارسی روی تمام متن‌های جست‌وجو
products_clean_df["search_text"] = products_clean_df["raw_search_text"].map(
    normalize_persian_text
)

# بررسی نهایی برای جلوگیری از متن‌های خالی
empty_search_text_count = (products_clean_df["search_text"].str.len() == 0).sum()
assert empty_search_text_count == 0, f"{empty_search_text_count:,} محصول متن جست‌وجوی خالی دارند!"

print(f"Prepared search text for: {len(products_clean_df):,} products")
print(f"Empty normalized search texts: {empty_search_text_count:,}")

print("\nSample before/after normalization:")
display(products_clean_df[["id", "title_fa", "raw_search_text", "search_text"]].head(3))


✓ Normalizer imported from src.digikala_ai.utils.normalization
Text columns used for retrieval: ['title_fa', 'Brand', 'Category1', 'Category2']
Prepared search text for: 948,352 products
Empty normalized search texts: 0

Sample before/after normalization:


,id,title_fa,raw_search_text,search_text
0,7096438,آبسلانگ مدل s5 بسته 250 عددی,آبسلانگ مدل s5 بسته 250 عددی متفرقه آبسلانگ,آبسلانگ مدل s5 بسته 250 عددی متفرقه آبسلانگ
1,2845119,آبسلانگ مدل M-1 بسته 400 عددی,آبسلانگ مدل M-1 بسته 400 عددی متفرقه آبسلانگ,آبسلانگ مدل M-1 بسته 400 عددی متفرقه آبسلانگ
2,6117745,آبسلانگ مدل m50 مجموعه 500 عددی,آبسلانگ مدل m50 مجموعه 500 عددی متفرقه آبسلانگ,آبسلانگ مدل m50 مجموعه 500 عددی متفرقه آبسلانگ


In [6]:
# اگر این import خطا داد، متن کامل خطا را بفرست؛
# فعلاً بستهٔ جدیدی نصب نکن.
from rank_bm25 import BM25Okapi
import re
import numpy as np

# ساده‌سازی و توکن‌سازی متن نرمال‌شده برای BM25
def tokenize_fa(text):
    return re.findall(r"\w+", str(text), flags=re.UNICODE)

# ساخت توکن‌ها و ایندکس BM25
bm25_corpus_tokens = [
    tokenize_fa(text)
    for text in products_clean_df["search_text"]
]

bm25_index = BM25Okapi(bm25_corpus_tokens)

# بررسی اولیهٔ ساختار ایندکس
assert len(bm25_corpus_tokens) == len(products_clean_df)
assert len(bm25_index.doc_len) == len(products_clean_df)

print(f"BM25 index built successfully.")
print(f"Indexed documents: {len(bm25_corpus_tokens):,}")
print(f"Average tokens per document: {np.mean(bm25_index.doc_len):.2f}")
print(f"Example tokens: {bm25_corpus_tokens[0][:15]}")


BM25 index built successfully.
Indexed documents: 948,352
Average tokens per document: 15.13
Example tokens: ['آبسلانگ', 'مدل', 's5', 'بسته', '250', 'عددی', 'متفرقه', 'آبسلانگ']


In [7]:
def search_bm25(query, top_k=5):
    query_normalized = normalize_persian_text(query)
    query_tokens = tokenize_fa(query_normalized)

    scores = bm25_index.get_scores(query_tokens)

    # انتخاب top-k بدون مرتب‌سازی کامل همهٔ محصولات
    candidate_indices = np.argpartition(scores, -top_k)[-top_k:]
    candidate_indices = candidate_indices[
        np.argsort(scores[candidate_indices])[::-1]
    ]

    results = products_clean_df.iloc[candidate_indices][
        ["id", "title_fa", "Brand", "Category1"]
    ].copy()
    results["score"] = scores[candidate_indices]

    return results.reset_index(drop=True)


# یک پرس‌وجوی نمونه برای بررسی عملکرد پایه
display(search_bm25("آبسلانگ مدل", top_k=5))


,id,title_fa,Brand,Category1,score
0,9855257,آبسلانگ مدل آلار بسته 40 عددی,متفرقه,آبسلانگ,19.904664
1,10392313,آبسلانگ مدل B10 بسته 300 عددی,متفرقه,آبسلانگ,19.904664
2,1452939,آبسلانگ مدل 1002 بسته 30 عددی,متفرقه,آبسلانگ,19.904664
3,8033206,آبسلانگ مدل Rsh0200 بسته 200 عددی,متفرقه,آبسلانگ,19.904664
4,7611799,آبسلانگ مدل T1 بسته 35 عددی,متفرقه,آبسلانگ,19.904664


In [8]:
from pathlib import Path
import os
import torch

# ریشه پروژه از Cell 1 استفاده می‌شود.
assert PROJECT_ROOT.is_dir(), f"Project root not found: {PROJECT_ROOT}"

MODEL_CACHE_DIR = PROJECT_ROOT / "artifacts" / "model_cache"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# مدل پایه مطابق ساختار موجود پروژه
MODEL_NAME = "BAAI/bge-m3"

print(f"Project root: {PROJECT_ROOT}")
print(f"Model cache directory: {MODEL_CACHE_DIR}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("Dense encoding will run on CPU.")


Project root: D:\goal\programming\quera\boot_camp\project3
Model cache directory: D:\goal\programming\quera\boot_camp\project3\artifacts\model_cache
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
GPU memory: 4.00 GB


In [11]:
from pathlib import Path
import torch
from sentence_transformers import SentenceTransformer

# تعیین مسیر ریشه پروژه و پوشه کش مدل‌ها
PROJECT_ROOT = Path.cwd().resolve().parent
MODEL_CACHE_DIR = PROJECT_ROOT / "artifacts" / "model_cache"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

SNAPSHOTS_DIR = MODEL_CACHE_DIR / "models--BAAI--bge-m3" / "snapshots"

# بررسی اینکه آیا اسنپ‌شات لوکال از قبل وجود دارد یا خیر
local_model_path = None
if SNAPSHOTS_DIR.exists():
    snapshot_candidates = sorted([p for p in SNAPSHOTS_DIR.iterdir() if p.is_dir()])
    if snapshot_candidates:
        local_model_path = snapshot_candidates[-1]

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Target device: {device}")
print(f"Model cache directory: {MODEL_CACHE_DIR}")

# بارگذاری مدل (لود لوکال در صورت وجود، دانلود یا استفاده از کش پیش‌فرض در غیراین‌صورت)
if local_model_path:
    print(f"Loading local BGE-M3 snapshot from: {local_model_path}")
    dense_model = SentenceTransformer(
        str(local_model_path),
        device=device,
        local_files_only=True,
        token=False
    )
else:
    print("Local snapshot not found. Loading BGE-M3 from HuggingFace/cache...")
    dense_model = SentenceTransformer(
        "BAAI/bge-m3",
        cache_folder=str(MODEL_CACHE_DIR),
        device=device,
        token=False
    )

print("\n--- Model Specifications ---")
print(f"Embedding Dimension: {dense_model.get_sentence_embedding_dimension()}")
print(f"Max Sequence Length: {dense_model.max_seq_length}")
print(f"Active Device: {dense_model.device}")


Target device: cuda
Model cache directory: D:\goal\programming\quera\boot_camp\project3\artifacts\model_cache
Loading local BGE-M3 snapshot from: D:\goal\programming\quera\boot_camp\project3\artifacts\model_cache\models--BAAI--bge-m3\snapshots\5617a9f61b028005a4858fdac845db406aefb181

--- Model Specifications ---
Embedding Dimension: 1024
Max Sequence Length: 8192
Active Device: cuda:0


In [14]:
import time
import torch

# استفاده از متغیر دقیق نوت‌بوک: products_clean_df
sample_texts = products_clean_df["search_text"].head(4).tolist()

print(f"Number of test texts: {len(sample_texts)}")
print(f"Sample text: {sample_texts[0]}")

# تنظیم حداکثر طول توالی متناسب با طول فیلدهای جست‌وجو (عنوان + برند + دسته‌ها)
dense_model.max_seq_length = 256

# پاک‌سازی حافظه آزاد GPU و ریست کردن آمار مصرف حافظه
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()

sample_embeddings = dense_model.encode(
    sample_texts,
    batch_size=4,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

elapsed_time = time.perf_counter() - start_time

print(f"\nEmbedding shape: {sample_embeddings.shape}")
print(f"Embedding dtype: {sample_embeddings.dtype}")
print(f"Elapsed time: {elapsed_time:.3f} seconds")

if torch.cuda.is_available():
    peak_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    print(f"Peak allocated GPU memory: {peak_memory_gb:.2f} GB")


Number of test texts: 4
Sample text: آبسلانگ مدل s5 بسته 250 عددی متفرقه آبسلانگ

Embedding shape: (4, 1024)
Embedding dtype: float32
Elapsed time: 0.693 seconds
Peak allocated GPU memory: 2.13 GB


In [15]:
import gc
import time
import torch

# نمونهٔ کمی بزرگ‌تر برای ارزیابی batch
benchmark_texts = products_clean_df["search_text"].head(128).tolist()

# مدل همچنان با حداکثر ۲۵۶ توکن برای هر متن کار می‌کند
dense_model.max_seq_length = 256

def benchmark_encoding(texts, batch_size):
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start_time = time.perf_counter()

    try:
        embeddings = dense_model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        elapsed_time = time.perf_counter() - start_time

        if torch.cuda.is_available():
            peak_memory_gb = (
                torch.cuda.max_memory_allocated() / (1024 ** 3)
            )
        else:
            peak_memory_gb = None

        return {
            "status": "success",
            "shape": embeddings.shape,
            "elapsed_seconds": round(elapsed_time, 3),
            "peak_memory_gb": (
                round(peak_memory_gb, 3)
                if peak_memory_gb is not None
                else None
            ),
        }

    except RuntimeError as error:
        if "out of memory" in str(error).lower():
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            return {
                "status": "CUDA out of memory",
                "shape": None,
                "elapsed_seconds": None,
                "peak_memory_gb": None,
            }

        raise

# ابتدا مقدار محتاطانه‌تر را بررسی می‌کنیم
result = benchmark_encoding(
    texts=benchmark_texts,
    batch_size=8,
)

print("Benchmark result:")
for key, value in result.items():
    print(f"{key}: {value}")


Benchmark result:
status: success
shape: (128, 1024)
elapsed_seconds: 1.02
peak_memory_gb: 2.139


In [16]:
import numpy as np
import pandas as pd

# ۱. انکود کردن کوئری تست (دقیقاً مشابه کوئری تست BM25)
test_query = "آبسلانگ مدل"
print(f"Encoding test query: '{test_query}'")

query_embedding = dense_model.encode(
    [test_query],
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,  # وقتی نرمالایز L2 شوند، ضرب داخلی معادل شباهت کسینوسی است
)

# ۲. امبدینگ‌های ۱۲۸ محصول نمونه از مرحله قبل (که نرمالایز شده بودند)
# محاسبه Dot Product بین بردار کوئری (1, 1024) و بردار محصولات (128, 1024)
test_products_df = products_clean_df.head(128).copy()

# برای اطمینان مجدد از امبدینگ‌های نمونه مرحله بنچمارک:
sample_embeddings_128 = dense_model.encode(
    test_products_df["search_text"].tolist(),
    batch_size=8,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# محاسبه شباهت کسینوسی (Cosine Similarity)
cosine_scores = np.dot(sample_embeddings_128, query_embedding.T).flatten()
test_products_df["dense_score"] = cosine_scores

# ۳. دریافت ۵ نتیجه برتر بر اساس بیشترین شباهت
top_k = 5
top_matches = test_products_df.sort_values(by="dense_score", ascending=False).head(top_k)

print(f"\nTop {top_k} Dense Retrieval results for query: '{test_query}'")
display_cols = ["id", "title_fa", "Brand", "Category1", "dense_score"]
top_matches[display_cols].reset_index(drop=True)


Encoding test query: 'آبسلانگ مدل'

Top 5 Dense Retrieval results for query: 'آبسلانگ مدل'


,id,title_fa,Brand,Category1,dense_score
0,8950128,آبسلانگ مدل چوبی بسته 30 عددی,متفرقه,آبسلانگ,0.608633
1,9855257,آبسلانگ مدل آلار بسته 40 عددی,متفرقه,آبسلانگ,0.602488
2,9861719,آبسلانگ مدل 01 بسته 30 عددی,متفرقه,آبسلانگ,0.597190
3,1912926,استند ابسلانگ مدل S01,متفرقه,آبسلانگ,0.596661
4,3901016,آبسلانگ مدل 01 بسته 50 عددی,متفرقه,آبسلانگ,0.593635


In [17]:
import json
from pathlib import Path
import faiss
import numpy as np
import pandas as pd

# ۱. تعیین مسیر ذخیره‌سازی Artifactها
PROJECT_ROOT = Path.cwd().resolve().parent
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "dense_baseline"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# ۲. انتخاب پیکره (برای شروع سریع‌تر و اطمینان از صحت پایپ‌لاین، فعلاً روی نمونه ۱۰۰۰ عددی یا کل داده)
# پوریا جان: اگر می‌خواهی کل داده‌های clean را embed کنی، خط زیر را غیرفعال کن.
# فعلاً برای تست استاندارد اولیه روی ۱۰۰۰ نمونهٔ رفرنس تنظیم می‌کنیم:
CORPUS_SAMPLE_SIZE = 1000
if CORPUS_SAMPLE_SIZE:
    corpus_df = products_clean_df.sample(
        n=CORPUS_SAMPLE_SIZE, random_state=42
    ).reset_index(drop=True)
    print(f"Using a controlled sample of {CORPUS_SAMPLE_SIZE} products for Dense baseline index.")
else:
    corpus_df = products_clean_df.reset_index(drop=True).copy()
    print(f"Using full cleaned catalog: {len(corpus_df):,} products.")

# ۳. تابع ساخت متن جست‌وجو برای هر محصول
def make_corpus_text(row):
    parts = []
    if pd.notna(row.get("title_fa")):
        parts.append(str(row["title_fa"]))
    if pd.notna(row.get("Brand")):
        parts.append(f"برند: {row['Brand']}")
    if pd.notna(row.get("Category1")):
        parts.append(f"دسته‌بندی: {row['Category1']}")
    if pd.notna(row.get("Category2")):
        parts.append(f"زیردسته: {row['Category2']}")
    return " - ".join(parts)

corpus_texts = corpus_df.apply(make_corpus_text, axis=1).tolist()

# ۴. تولید Embeddingها با مدل BGE-M3 روی GPU
print("Encoding corpus texts with BGE-M3...")
dense_model.max_seq_length = 256

corpus_embeddings = dense_model.encode(
    corpus_texts,
    batch_size=64 if device == "cuda" else 16,
    show_progress_bar=True,
    normalize_embeddings=True,  # نرمال‌سازی برای استفاده از Cosine Similarity در FAISS
    convert_to_numpy=True,
)

corpus_embeddings = corpus_embeddings.astype(np.float32)
embedding_dim = corpus_embeddings.shape[1]

# ۵. ساخت ایندکس FAISS (IndexFlatIP برای Inner Product معادل Cosine Similarity روی بردار نرمال‌شده)
print("Building FAISS index...")
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(corpus_embeddings)

# ۶. ذخیره‌سازی Artifactها در پوشه مشخص‌شده
faiss.write_index(faiss_index, str(ARTIFACTS_DIR / "products.index.faiss"))
np.save(ARTIFACTS_DIR / "products.embeddings.npy", corpus_embeddings)
corpus_df[["id", "title_fa"]].to_parquet(ARTIFACTS_DIR / "products.index_map.parquet", index=False)

metadata = {
    "model_name": "BAAI/bge-m3",
    "corpus_size": len(corpus_df),
    "embedding_dim": int(embedding_dim),
    "normalized_embeddings": True,
    "similarity": "cosine via FAISS IndexFlatIP",
    "batch_size": 64 if device == "cuda" else 16,
    "product_id_order": "products.index_map.parquet row order"
}

with open(ARTIFACTS_DIR / "build_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("\n--- Dense Baseline Index Built Successfully ---")
print(f"FAISS vectors added: {faiss_index.ntotal:,}")
print(f"Artifacts saved to: {ARTIFACTS_DIR}")


Using a controlled sample of 1000 products for Dense baseline index.
Encoding corpus texts with BGE-M3...


Batches: 100%|██████████| 16/16 [00:07<00:00,  2.21it/s]

Building FAISS index...

--- Dense Baseline Index Built Successfully ---
FAISS vectors added: 1,000
Artifacts saved to: D:\goal\programming\quera\boot_camp\project3\artifacts\dense_baseline


In [18]:
# بارگذاری مجدد نقشه شناسه محصولات جهت اطمینان از استقلال کامل مرحله جست‌وجو
index_map_df = pd.read_parquet(ARTIFACTS_DIR / "products.index_map.parquet")

def search_dense(query: str, top_k: int = 5):
    """
    جست‌وجوی معنایی محصولات با مدل BGE-M3 و ایندکس FAISS
    """
    # ۱. تبدیل کوئری به بردار نرمال‌شده
    query_vec = dense_model.encode(
        [query],
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype(np.float32)
    
    # ۲. جست‌وجوی نزدیک‌ترین بردارها با FAISS
    scores, indices = faiss_index.search(query_vec, top_k)
    
    # ۳. استخراج اطلاعات محصولات متناظر
    matched_indices = indices[0]
    matched_scores = scores[0]
    
    results = []
    for rank, (idx, score) in enumerate(zip(matched_indices, matched_scores), start=1):
        if idx == -1: # در صورتی که نتیجه‌ای پیدا نشود
            continue
        product_info = index_map_df.iloc[idx]
        results.append({
            "rank": rank,
            "id": product_info["id"],
            "title_fa": product_info["title_fa"],
            "score": round(float(score), 4)
        })
        
    return pd.DataFrame(results)

# تست اولیه با یک کوئری عمومی
test_query = "گوشی موبایل سامسونگ"
results_df = search_dense(test_query, top_k=5)

print(f"Dense Search Results for: '{test_query}' (Index Size: {faiss_index.ntotal})")
results_df


Dense Search Results for: 'گوشی موبایل سامسونگ' (Index Size: 1000)


,rank,id,title_fa,score
0,1,8769905,سوهان ناخن مندیکس مدل قلب کد 1062 به همراه آین...,0.4454
1,2,8781964,سوهان ناخن مندیکس مدل کارتونی کد 843 به همراه ...,0.4403
2,3,5394113,کبریت مدل سیمپسون کد kbt2007,0.4160
3,4,11627522,محافظ صفحه نمایش شیشه‌ای سوپرسی مدل SH-SC مناس...,0.4059
4,5,6338742,سوهان ناخن مدل کارتونی کد 852,0.4036


In [22]:
import subprocess
import sys
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().resolve().parent
script_path = PROJECT_ROOT / "create_evaluation_queries.py"

print(f"Running generator script: {script_path.name}...")

# اجرای اسکریپت سازنده کوئری‌ها در ریشه پروژه
result = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8"
)

if result.returncode == 0:
    print("Script output:")
    print(result.stdout)
    
    # تأیید وجود و بارگذاری فایل تولید شده
    eval_file = PROJECT_ROOT / "evaluation" / "evaluation_queries.json"
    if eval_file.exists():
        with open(eval_file, "r", encoding="utf-8") as f:
            eval_data = json.load(f)
        
        print(f"✓ Successfully created and verified: {eval_file.name}")
        queries = eval_data.get("queries", [])
        print(f"Total queries generated: {len(queries)}")
        print("\nFirst query details:")
        print(json.dumps(queries[0], ensure_ascii=False, indent=2))
    else:
        print("Warning: Script succeeded but evaluation_queries.json not found in evaluation/ folder.")
else:
    print("Error running script:")
    print(result.stderr)


Running generator script: create_evaluation_queries.py...
Script output:
Wrote 25 queries to D:\goal\programming\quera\boot_camp\project3\evaluation\evaluation_queries.json

✓ Successfully created and verified: evaluation_queries.json
Total queries generated: 25

First query details:
{
  "id": "PD-01",
  "task_type": "product_discovery",
  "query": "گوشی سامسونگ سری S با قیمت مناسب",
  "language": "fa",
  "features": [
    "brand",
    "series",
    "price_intent"
  ],
  "note": "confirm relevant catalog products exist; broad intent needs relevance judgments"
}


In [24]:
# انتخاب ۵ کوئری اول برای بررسی کیفی عملکرد مدل معنایی
sample_queries = queries[:5]

print(f"Running qualitative Dense retrieval check for {len(sample_queries)} queries against our 1,000-sample FAISS index:\n")
print("=" * 80)

qualitative_results = []

for q_item in sample_queries:
    qid = q_item["id"]
    qtext = q_item["query"]
    task_type = q_item["task_type"]
    
    # جست‌وجو در ایندکس FAISS
    top_results_df = search_dense(qtext, top_k=3)
    
    print(f"\n[Query ID: {qid}] ({task_type})")
    print(f"Query Text: '{qtext}'")
    print("-" * 50)
    
    for _, row in top_results_df.iterrows():
        print(f"  Rank {row['rank']}: [ID: {row['id']}] Score: {row['score']:.4f} | {row['title_fa'][:60]}...")
        
    qualitative_results.append({
        "id": qid,
        "query": qtext,
        "task_type": task_type,
        "retrieved_top3": top_results_df[["id", "title_fa", "score"]].to_dict(orient="records")
    })

print("\n" + "=" * 80)
print("✓ Qualitative inspection finished.")


Running qualitative Dense retrieval check for 5 queries against our 1,000-sample FAISS index:


[Query ID: PD-01] (product_discovery)
Query Text: 'گوشی سامسونگ سری S با قیمت مناسب'
--------------------------------------------------
  Rank 1: [ID: 9850702] Score: 0.4565 | برچسب کنسول بازی ایکس باکس series s مدل color کد 1 مجموعه 4 ...
  Rank 2: [ID: 11627522] Score: 0.4444 | محافظ صفحه نمایش شیشه‌ای سوپرسی مدل SH-SC مناسب برای ساعت هو...
  Rank 3: [ID: 7019494] Score: 0.4429 | پیرسینگ گوش مدل شش ضلعی کد S...

[Query ID: PD-02] (product_discovery)
Query Text: 'یه گوشی شیائومی با باتری قوی و شارژدهی طولانی'
--------------------------------------------------
  Rank 1: [ID: 9258132] Score: 0.4332 | برچسب ماهوت طرح Army_Green مناسب برای ساعت هوشمند هوآوی Watc...
  Rank 2: [ID: 9261131] Score: 0.4200 | برچسب ماهوت طرح Red_Printed_Circuit_Board مناسب برای ساعت هو...
  Rank 3: [ID: 7144854] Score: 0.4191 | محافظ صفحه نمایش آکوییلا مدل AAGM مناسب برای ساعت هوشمند امی...

[Query ID: PD-03] (produ

In [25]:
import json
from datetime import datetime

# مسیر فایل خروجی طبق ساختار پوشه evaluation
output_eval_path = PROJECT_ROOT / "evaluation" / "dense_baseline_results.json"

results_payload = {
    "generated_at": datetime.now().isoformat(),
    "model_name": "BAAI/bge-m3",
    "embedding_dimension": 1024,
    "similarity_metric": "cosine (IndexFlatIP with L2 norm)",
    "index_corpus_size": len(corpus_df),
    "total_corpus_available": len(products_clean_df),
    "note": "Qualitative evaluation of Dense baseline on a controlled sample of 1,000 products.",
    "evaluation_results": qualitative_results
}

with open(output_eval_path, "w", encoding="utf-8") as f:
    json.dump(results_payload, f, ensure_ascii=False, indent=2)

print(f"✓ Results successfully saved to: {output_eval_path.relative_to(PROJECT_ROOT)}")
print(f"File size: {output_eval_path.stat().st_size:,} bytes")


✓ Results successfully saved to: evaluation\dense_baseline_results.json
File size: 4,499 bytes
